<a href="https://colab.research.google.com/github/acastellanos-ie/NLP-MBDS-EN/blob/main/07_rag/information_retrieval_vsm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Information Retrieval: Lexical and Dense Search

A retrieval system receives a query and ranks documents. It does not answer the question yet.

We will compare TF-IDF, BM25 and dense embeddings on the same small corpus. The examples are designed to show where exact terms help, where paraphrases break lexical search and why every ranking still needs evaluation.

In [ ]:
# @title Setup
%pip install -q "scikit-learn==1.7.1" "rank-bm25==0.2.2" "sentence-transformers==5.1.0"

## The corpus

Documents D0 and D1 describe almost the same event with different vocabulary. The other documents give us nearby and unrelated topics.

In [ ]:
documents = [
    "The company increased employee salaries after a strong quarter.",
    "The firm raised workers' pay following good results.",
    "The company opened a new office in Madrid.",
    "Employees requested flexible working hours.",
    "The ocean appears blue because water absorbs red light.",
    "A quick brown fox jumps over a lazy dog.",
]

for document_id, document in enumerate(documents):
    print(f"D{document_id}: {document}")

## Tokenization for lexical retrieval

TF-IDF and BM25 work with terms. Use the same lowercase word tokenizer for documents and queries, and remove common English stopwords.

BM25 expects a list of tokens for each document. Passing complete strings would make it iterate over characters instead of words.

In [ ]:
import re
import numpy as np
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stopwords = set(ENGLISH_STOP_WORDS)

def tokenize(text):
    words = re.findall(r"[a-z]+", text.lower())
    return [word for word in words if word not in stopwords]

tokenized_documents = [tokenize(document) for document in documents]
for document_id, tokens in enumerate(tokenized_documents):
    print(f"D{document_id}: {tokens}")

The output is a list of word tokens per document. This is the structure BM25 needs. It also makes the limitation visible: *company*, *firm* and *business* remain three unrelated strings.

## TF-IDF and BM25

TF-IDF represents each document as a weighted term vector and uses cosine similarity. BM25 uses term frequency, inverse document frequency and document-length normalization. Their scores use different scales, so compare rankings rather than raw TF-IDF and BM25 values.

In [ ]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_vectorizer = TfidfVectorizer(
    tokenizer=tokenize, token_pattern=None, lowercase=False, norm="l2"
)
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)
bm25 = BM25Okapi(tokenized_documents)

def tfidf_scores(query):
    query_vector = tfidf_vectorizer.transform([query])
    return cosine_similarity(query_vector, tfidf_matrix)[0]

def bm25_scores(query):
    return np.asarray(bm25.get_scores(tokenize(query)))

def top_indices(scores, k=3):
    return np.argsort(scores)[::-1][:k]

def print_ranking(scores, k=3):
    for rank, document_id in enumerate(top_indices(scores, k), start=1):
        print(f"{rank}. D{document_id} | score={scores[document_id]:.4f} | {documents[document_id]}")

### Exact terms

Start with a query that shares two terms with D0.

In [ ]:
lexical_query = "employee salaries"
lexical_tfidf = tfidf_scores(lexical_query)
lexical_bm25 = bm25_scores(lexical_query)

print("TF-IDF")
print_ranking(lexical_tfidf)
print("\nBM25")
print_ranking(lexical_bm25)

Both methods rank D0 first because it contains *employee* and *salaries*. D1 describes a similar event, but *workers* and *pay* do not match those query terms. Lexical retrieval is strong when the vocabulary overlaps.

### A paraphrase with no shared terms

Now ask for the same topic using *business*, *staff* and *compensation*.

In [ ]:
paraphrase_query = "How did the business improve staff compensation?"
paraphrase_tfidf = tfidf_scores(paraphrase_query)
paraphrase_bm25 = bm25_scores(paraphrase_query)

print(f"Query tokens: {tokenize(paraphrase_query)}")
print(f"Maximum TF-IDF score: {paraphrase_tfidf.max():.4f}")
print(f"Maximum BM25 score:    {paraphrase_bm25.max():.4f}")
print("\nThe displayed order is only a tie between zero scores:")
print_ranking(paraphrase_tfidf)

Every lexical score is zero. The printed document order is not a meaningful ranking; it is just how the sorting code breaks a tie. This is more informative than saying the method returned a wrong top document: with this vocabulary, it had no matching signal at all.

## Dense retrieval

Use the sentence embedding model from the semantics notebook. Documents and queries are compared by cosine similarity in the learned vector space.

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model_id = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(embedding_model_id)
document_embeddings = embedding_model.encode(documents, normalize_embeddings=True)

def dense_scores(query):
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)[0]
    return document_embeddings @ query_embedding

paraphrase_dense = dense_scores(paraphrase_query)
print_ranking(paraphrase_dense)

Dense retrieval ranks D1 and D0 first even though the query shares no content words with them. The embedding model connects *business/company/firm*, *staff/workers/employee* and *compensation/pay/salaries*.

This is one small, constructed example. It shows a capability, not that dense retrieval is always better. Exact names, codes and rare terms can still favour lexical methods.

## What if the corpus has no answer?

A ranking function always has a first document. That does not mean the first document is relevant.

In [ ]:
missing_query = "What is the CEO name?"
missing_tfidf = tfidf_scores(missing_query)
missing_dense = dense_scores(missing_query)

print("TF-IDF")
print_ranking(missing_tfidf, k=1)
print("\nDense")
print_ranking(missing_dense, k=1)

TF-IDF returns a zero-score tie. Dense retrieval still returns D1 with a low positive score because it must rank something. Neither result contains a CEO name. Abstention requires a calibrated threshold or a later component that checks whether the evidence supports an answer.

## A small evaluation

Define relevant documents for three answerable queries and calculate Hit@2. This only checks whether at least one relevant document appears in the first two positions.

In [ ]:
evaluation_queries = [
    ("employee salaries", {0, 1}),
    ("How did the business improve staff compensation?", {0, 1}),
    ("Why is the sea blue?", {4}),
]

retrievers = {
    "TF-IDF": tfidf_scores,
    "BM25": bm25_scores,
    "Dense": dense_scores,
}

hit_at_2 = {}
for name, score_function in retrievers.items():
    hits = []
    for query, relevant_ids in evaluation_queries:
        retrieved_ids = set(top_indices(score_function(query), k=2))
        hits.append(bool(retrieved_ids & relevant_ids))
    hit_at_2[name] = sum(hits) / len(hits)
    print(f"{name:<7} Hit@2 = {hit_at_2[name]:.3f} | per query: {hits}")

Dense retrieval hits a relevant document for all three queries. TF-IDF and BM25 miss the paraphrase and score 2 out of 3.

The corpus and queries were created for this demonstration, so this is a code check rather than a benchmark. A real comparison needs many queries, relevance judgements and metrics such as Recall@k, MRR or nDCG.

In [ ]:
# @title Consistency checks
assert isinstance(tokenized_documents[0], list)
assert top_indices(lexical_tfidf, 1)[0] == 0
assert paraphrase_tfidf.max() == 0
assert paraphrase_bm25.max() == 0
assert list(top_indices(paraphrase_dense, 2)) == [1, 0]
assert hit_at_2 == {"TF-IDF": 2 / 3, "BM25": 2 / 3, "Dense": 1.0}
print("Checks passed: notebook text and outputs are consistent.")

# Takeaway

- TF-IDF and BM25 are strong when queries and documents share informative terms.
- Dense retrieval can connect paraphrases with little or no lexical overlap.
- The score scales of different retrieval methods are not directly comparable.
- A top-ranked document is not automatically relevant, especially when the answer is absent.
- Retrieval quality must be evaluated before adding a generator.